# Clase 12 — Innovación responsable e inteligencia artificial (IA): ética, sesgo e interpretabilidad

Cuaderno de clase: lo proyectas y ejecutas en vivo. Alterna una explicación teórica breve con celdas de código funcionando.

## Objetivos de aprendizaje

- Reconoces cómo se cuela un sesgo algorítmico en un sistema de IA sin que nadie lo busque a propósito
- Construyes un dataset sintético con dos grupos demográficos donde uno arrastra una desventaja histórica
- Entrenas un clasificador simple con scikit-learn sobre ese dataset
- Calculas una métrica de equidad estándar, la diferencia de oportunidad igual, y la interpretas
- Visualizas la brecha entre grupos con un gráfico de barras

## Cuando un modelo aprende un sesgo

Ningún equipo diseña un sistema para discriminar. El sesgo entra por la puerta trasera: datos poco representativos, etiquetas que heredan prejuicios históricos y métricas de éxito que no incluyen la equidad como objetivo.

Casos reales que ya conoces del manual:

- El sistema COMPAS (Perfilado de Gestión de Delincuentes Correccionales para Sanciones Alternativas), usado en tribunales de Estados Unidos, marcaba a personas afrodescendientes como de alto riesgo con una tasa de falsos positivos muy superior a la de la población blanca
- En 2015, el reconocimiento de imágenes de Google Photos etiquetó fotografías de personas negras como "gorilas"
- Herramientas de búsqueda de empleo mostraron anuncios de puestos bien remunerados a hombres con mucha más frecuencia que a mujeres

En los tres casos, el proceso técnico funcionaba "bien" según su métrica de entrenamiento. El fallo estaba en los datos y en qué se decidió medir. En este cuaderno reproduces ese patrón a pequeña escala: generas un dataset con un sesgo de representación deliberado, entrenas un modelo sobre él y mides cuánto se nota.

## Genera un dataset sintético con dos grupos

Creas un problema de clasificación binaria con dos grupos demográficos ficticios, A y B. El proceso causal que genera el resultado real es idéntico en ambos grupos, pero los datos de origen del grupo B reflejan valores sistemáticamente más bajos en una variable clave y más ruido de medición en la etiqueta. Así simulas un sesgo de representación en el origen de los datos, no un sesgo introducido por el modelo.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n_por_grupo = 1000

# --- Grupo A: grupo mayoritario, bien representado en el histórico ---
edad_a = rng.normal(40, 10, n_por_grupo)
ingresos_a = rng.normal(50, 15, n_por_grupo)
antiguedad_a = rng.normal(8, 4, n_por_grupo)
score_a = 0.04 * ingresos_a + 0.3 * antiguedad_a - 5
etiqueta_a = (score_a + rng.normal(0, 3, n_por_grupo) > 0).astype(int)

# --- Grupo B: grupo minoritario, peor representado en el histórico ---
# El proceso causal real es el mismo, pero los datos disponibles reflejan
# ingresos declarados sistemáticamente más bajos y una etiqueta con más
# ruido de medición: así se simula el sesgo de representación en el origen
# de los datos, no un sesgo del modelo en sí mismo.
edad_b = rng.normal(38, 10, n_por_grupo)
ingresos_b = rng.normal(38, 15, n_por_grupo)
antiguedad_b = rng.normal(6, 4, n_por_grupo)
score_b = 0.04 * ingresos_b + 0.3 * antiguedad_b - 5
etiqueta_b = (score_b + rng.normal(0, 5, n_por_grupo) > 0).astype(int)

df = pd.concat([
    pd.DataFrame({"edad": edad_a, "ingresos": ingresos_a, "antiguedad": antiguedad_a,
                  "grupo": "A", "etiqueta": etiqueta_a}),
    pd.DataFrame({"edad": edad_b, "ingresos": ingresos_b, "antiguedad": antiguedad_b,
                  "grupo": "B", "etiqueta": etiqueta_b}),
], ignore_index=True)

df.sample(5, random_state=1)

## Entrena un clasificador simple

Ajustas una regresión logística (`LogisticRegression` de scikit-learn) sobre el dataset combinado, sin decirle nada sobre el grupo demográfico: esa columna no entra como variable de entrada. El modelo no "sabe" que existen dos grupos; aun así, como ves a continuación, puede tratarlos de forma distinta.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X = df[["edad", "ingresos", "antiguedad"]].values
y = df["etiqueta"].values
grupo = df["grupo"].values

X_train, X_test, y_train, y_test, grupo_train, grupo_test = train_test_split(
    X, y, grupo, test_size=0.3, random_state=42, stratify=grupo
)

modelo = LogisticRegression()
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)

resultados = pd.DataFrame({"grupo": grupo_test, "y_real": y_test, "y_pred": y_pred})
resultados.head()

## Audita la equidad del modelo

La tasa de verdaderos positivos (TPR) mide qué proporción de los casos positivos reales identifica correctamente el modelo. La diferencia de oportunidad igual (equal opportunity difference) compara la TPR entre grupos y resta el valor más bajo al más alto.

Un valor cercano a cero indica que el modelo detecta con la misma eficacia a los positivos reales de ambos grupos. Un valor alejado de cero señala que uno de ellos recibe sistemáticamente menos verdaderos positivos que el otro, aunque el algoritmo nunca haya visto esa columna.

In [ ]:
import matplotlib.pyplot as plt


def tasa_verdaderos_positivos(subconjunto):
    """TPR = proporción de positivos reales que el modelo detecta como positivos."""
    positivos_reales = subconjunto[subconjunto["y_real"] == 1]
    if len(positivos_reales) == 0:
        return np.nan
    return (positivos_reales["y_pred"] == 1).mean()


def diferencia_oportunidad_igual(df_resultados, columna_grupo="grupo"):
    """Calcula la TPR por grupo y la diferencia entre el valor más alto y el
    más bajo: es la métrica de equal opportunity difference."""
    tprs = df_resultados.groupby(columna_grupo).apply(tasa_verdaderos_positivos)
    return tprs, tprs.max() - tprs.min()


tprs_por_grupo, diferencia = diferencia_oportunidad_igual(resultados)

print("TPR por grupo:")
print(tprs_por_grupo)
print(f"\nDiferencia de oportunidad igual: {diferencia:.3f}")

fig, ax = plt.subplots(figsize=(6, 5))
colores = ["#4C72B0", "#DD8452"]
barras = ax.bar(tprs_por_grupo.index, tprs_por_grupo.values, color=colores, width=0.5)

ax.set_ylim(0, 1)
ax.set_ylabel("Tasa de verdaderos positivos (TPR)")
ax.set_title("Auditoría de equidad: TPR por grupo demográfico")
ax.axhline(y=tprs_por_grupo.max(), linestyle="--", color="gray", alpha=0.5)

for barra, valor in zip(barras, tprs_por_grupo.values):
    ax.text(barra.get_x() + barra.get_width() / 2, valor + 0.02, f"{valor:.2f}", ha="center")

ax.text(0.5, -0.12, f"Diferencia de oportunidad igual = {diferencia:.3f}",
        transform=ax.transAxes, ha="center", fontsize=10, style="italic")

plt.tight_layout()
plt.show()

## Resumen de puntos clave

- El sesgo algorítmico casi nunca es intencional: nace en los datos (representación, ruido de medición, etiquetas heredadas) y se manifiesta en las decisiones del modelo
- Un modelo puede tratar a un grupo peor que a otro aunque nunca reciba esa variable como entrada: el sesgo se filtra a través de factores correlacionados
- La diferencia de oportunidad igual compara la tasa de verdaderos positivos (TPR) entre grupos y te dice si el modelo detecta con la misma eficacia a quienes de verdad cumplen la condición positiva
- Ninguna métrica de equidad es universalmente correcta: elige la que corresponda al daño concreto que quieres evitar en tu caso de uso
- Auditar un modelo por grupo demográfico no es un paso opcional al final del proyecto: forma parte del control de calidad, igual que la exactitud o la precisión

**Abre ahora el cuaderno de práctica companion, `../notebooks_practica/clase12_practica_innovacion_responsable_ia.ipynb`, y pon a prueba lo que acabas de ver.**